# Process Landsat-8 annual cloud-free composite from Google Earth Engine
This code snippet provides an example of accessing Landsat-8 data from Google Earth Engine, compositing cloud-free images over a year to produce an annual cloud-free composite, using the earthengine-api for Python (read the [docs](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api)). We will use the server side processing of GEE.

You will first need to register for a (free) Google Cloud project account.

We will use the [USGS Landsat 8 Level 2, Collection 2, Tier 1](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2) atmospherically corrected surface reflectance data.

You can browse other available datasets in the [Earth Engine Catalog](https://developers.google.com/earth-engine/datasets/catalog). There is also the [awesome-gee-community-catalog](https://gee-community-catalog.org/projects/) providing additional datasets contributed by the open-source community.

### Acknowledgements

This code was contributed by Kira Baker and published for the [SARP snippets](https://github.com/NASA-SARP/2026-Coding-Alex/tree/main/snippets) by Alex Saunders.

## Setup

In [1]:
# Install earthengine-api
# !pip install earthengine-api
# !pip install geemap
# !pip install geedim
# !pip install --upgrade --pre xee

In [2]:
from pathlib import Path
import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import box

# Only needed if streaming into xarray
# import xee
# from xee import helpers
# import xarray as xr

In [3]:
# Trigger the authentication flow
ee.Authenticate()

# Initialize the library
# You will need to follow the link, then copy and paste the verification code
ee.Initialize() #project='my-project'

## Define inputs

In [4]:
# Define collection name
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
bands = [f'SR_B{i}' for i in range(1,8)]
print('Bands:', bands)

# Define year to create the composite for
year = 2025

# Define region of interest and create a shapely box
region = box(-117.5, 32, -116, 33.5) #xmin, ymin, xmax, ymax

Bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']


## Define functions to work on the images
* Mask clouds using the QA_PIXEL band: "Pixel quality attributes generated from the CFMASK algorithm."
* Apply scaling and offset parameters to convert to surface reflectance (0-1)

In [5]:
# Apply conversion for optical bands
def scale_image(image):   
    return image.select(bands).multiply(0.0000275).add(-0.2)

# Cloud masking function
def mask_clouds(image):

    # Get the QA band
    qa = image.select('QA_PIXEL')

    # Create the mask - fill, dilated cloud, cirrus, cloud, cloud shadow
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)        # Fill
        .And(qa.bitwiseAnd(1 << 1).eq(0))  # Dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # Cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # Cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # Cloud shadow
    )
    
    return image.updateMask(mask)

## Filter the image collection for the region and dates

In [6]:
# Set up the start and end dates for the year
start_date = ee.Date.fromYMD(year, 1, 1)
end_date = start_date.advance(1, 'year')

# Convert the region to ee feature collection
region_gdf = gpd.GeoDataFrame(geometry=[region], crs=4326)
region_geometry = geemap.geopandas_to_ee(region_gdf).geometry()

# Filter collection for region and dates
collection = landsat8.filterBounds(region_geometry).filterDate(start_date, end_date)

# Get size and dates of filtered collection
ic_size = collection.size().getInfo()
image_ids = collection.aggregate_array('system:index').getInfo()
image_dates = pd.to_datetime(np.unique([item.split('_')[-1] for item in image_ids]))
print(f'Collection returned {ic_size} images across {len(image_dates)} unique image dates in {year}')

Collection returned 87 images across 44 unique image dates in 2025


## Access and process Landsat-8 data to create an annual cloud-free composite

* Get Landsat-8 surface reflectance images from the year
* Apply cloud masking
* Apply band scaling and offset parameters to retrieve surface reflectance values
* Create a median composite from all cloud-free images in the year

In [7]:
# Test cloud mask and surf refl conversion on one image
image = collection.first()
image_cloudmasked = scale_image(mask_clouds(image))

image_id = image.get('system:index').getInfo()
image_date = str(pd.to_datetime(image_id.split('_')[-1]).date())

In [8]:
# Run the cloud mask and surf refl conversions on all images
# Clip each image to the region of interest
image_stack = (collection.map(mask_clouds)
                       .map(scale_image)
                       .map(lambda image: image.clip(region_geometry)))

# Create median composite of all cloud-free pixels 
composite = ee.Image(image_stack.median())

### Map plotting

In [12]:
Map = geemap.Map()

l8_viz = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.0, 'max': 0.3}

Map.addLayer(region_geometry, {'color': 'FF0000'}, ' Region')

# Add the test single cloud masked image
image_name = f'{image_date}'
Map.addLayer(image_cloudmasked, l8_viz, image_name, shown=False)

# Add the annual median composite
composite_name = f'Composite {year}'
Map.addLayer(composite, l8_viz, composite_name)

Map.centerObject(region_geometry, 9)

Map

Map(center=[32.75012774342115, -116.74999999999996], controls=(WidgetControl(options=['position', 'transparent…

## Data streaming and download
For code to stream the data into an xarray DataArray or download the data see the [downloading Google Earth Engine data snippet.](https://github.com/NASA-SARP/2026-Coding-Alex/blob/main/snippets/download_earthengine_data.ipynb)

In [18]:
# Function to get stats for the annual CDL multiband image
# Add stats to each feature in the feature collection
def add_stats_per_feature(feature):
    stats = almond_multiband.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry(),
        scale=30,
        bestEffort=True)
    return feature.set(stats)

# Run the add_stats function on the counties
# For each county feature, adds almond mean (% area) stats for each band (year) in the multiband image
ca_counties_stats = ca_counties.map(add_stats_per_feature)

# Convert from ee feature collection to pandas geodataframe
ca_counties_stats_gdf = geemap.ee_to_gdf(ca_counties_stats)
ca_counties_stats_gdf.head()

NameError: name 'ca_counties' is not defined